# **BERT for Sarcasm Detection**

## Objective

This notebook implements a fine-tuned BERT model for sarcasm detection across multiple datasets:

- News Headlines Dataset
- Reddit SARC Dataset
- SemEval Irony Dataset

## Goals

- Fine-tune BERT for in-domain sarcasm detection (RQ1)
- Evaluate whether contextual information improves detection (RQ2)
- Assess cross-domain generalisation (RQ3)

## Workflow

1. Clone repository and install dependencies
2. Load preprocessed datasets
3. RQ1 — In-domain BERT (no context)
4. RQ2 — BERT with vs without context (Reddit)
5. RQ3 — Cross-domain generalisation
6. Full results summary

## 1. Setup

In [1]:
import os
import sys
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

BASE_DIR = os.path.dirname(os.getcwd())
PROC_DIR = os.path.join(BASE_DIR, "data", "processed")
MODELS_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

sys.path.append(os.path.join(os.getcwd(), ".."))

from src.deep_models import get_bert_model,tokenize_data,make_dataloader,train_bert,predict_bert,DualEncoderBert,DualEncoderDataset,train_dual_encoder,predict_dual_encoder
from src.evaluation import evaluate
from transformers import BertForSequenceClassification, BertTokenizer
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

SEEDS = [42, 123, 456, 789, 1011, 1213, 1415, 1617, 1819, 2021]
print(f"\nWill run {len(SEEDS)} seeds: {SEEDS}")

GPU available: True
Device: NVIDIA GeForce RTX 5070 Ti Laptop GPU

Will run 10 seeds: [42, 123, 456, 789, 1011, 1213, 1415, 1617, 1819, 2021]


In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## 2. Load Preprocessed Datasets

In [4]:
hl_train = pd.read_csv(os.path.join(PROC_DIR, "headlines_train.csv"))
hl_test = pd.read_csv(os.path.join(PROC_DIR, "headlines_test.csv"))

rd_train_full = pd.read_csv(os.path.join(PROC_DIR, "reddit_train.csv"))
rd_test_full = pd.read_csv(os.path.join(PROC_DIR, "reddit_test.csv"))

se_train = pd.read_csv(os.path.join(PROC_DIR, "semeval_train.csv"))
se_test = pd.read_csv(os.path.join(PROC_DIR, "semeval_test.csv"))

print("Dataset sizes:")
print(f"  Headlines train: {len(hl_train):>6}  test: {len(hl_test)}")
print(f"  Reddit train: {len(rd_train_full):>6}  test: {len(rd_test_full)}")
print(f"  SemEval train: {len(se_train):>6}  test: {len(se_test)}")

Dataset sizes:
  Headlines train:  22895  test: 5724
  Reddit train: 808610  test: 202153
  SemEval train:   3052  test: 764


## 4. RQ1 — In-Domain BERT (No Context)

A fresh BERT model is fine-tuned separately on each dataset and
evaluated on the held-out test set of the same dataset.

This answers: **How effectively can BERT detect sarcasm in short informal text?**

Fine-tuned models are saved to disk for reuse in RQ3.

In [8]:
# ============================================================
# 4. RQ1 — IN-DOMAIN BERT (NO CONTEXT), 10 SEEDS
# ============================================================
print("=" * 60)
print("RQ1: IN-DOMAIN BERT — NO CONTEXT")
print("=" * 60)

# ---- Load existing results if resuming ----
rq1_results_path = os.path.join(MODELS_DIR, "rq1_bert_results.csv")

if os.path.exists(rq1_results_path):
    rq1_df_existing  = pd.read_csv(rq1_results_path)
    rq1_all_results  = rq1_df_existing.to_dict("records")
    completed_rq1    = [
        (r["dataset"], r["seed"]) for r in rq1_all_results
    ]
    print(f"Resuming — {len(completed_rq1)} runs already completed")
    print(f"Completed: {completed_rq1}")
else:
    rq1_all_results = []
    completed_rq1   = []
    print("Starting fresh run")

for seed in SEEDS:
    print(f"\n{'='*40}")
    print(f"SEED {seed}")
    print(f"{'='*40}")

    set_seed(seed)

    # Subsample Reddit with this seed
    rd_train = rd_train_full.sample(20000, random_state=seed)
    rd_test  = rd_test_full.sample(5000,  random_state=seed)

    datasets_rq1 = [
        ("Headlines", hl_train, hl_test, "text"),
        ("Reddit",    rd_train, rd_test, "text"),
        ("SemEval",   se_train, se_test, "text"),
    ]

    for dataset_name, train_df, test_df, text_col in datasets_rq1:

        # Skip if already completed
        if (dataset_name, seed) in completed_rq1:
            print(f"Skipping {dataset_name} seed {seed} — already done")
            continue

        print(f"\n--- {dataset_name} ---")

        set_seed(seed)  # reset before each model init for independence
        model, tokenizer = get_bert_model()

        train_enc    = tokenize_data(train_df[text_col], tokenizer)
        test_enc     = tokenize_data(test_df[text_col],  tokenizer)
        train_loader = make_dataloader(train_enc, train_df["label"], shuffle=True)
        test_loader  = make_dataloader(test_enc,  test_df["label"],  shuffle=False)

        model  = train_bert(model, train_loader, epochs=3)
        y_pred = predict_bert(model, test_loader)
        y_true = test_df["label"].values

        result = evaluate(y_true, y_pred,
                          dataset_name=dataset_name,
                          model_name="BERT (no context)")
        result["seed"] = seed
        rq1_all_results.append(result)

        # Save model per seed — reused in RQ3
        save_path = os.path.join(
            MODELS_DIR, f"bert_{dataset_name.lower()}_seed{seed}"
        )
        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"Model saved: {save_path}")

        # Save results after every dataset — granular checkpoint
        rq1_df = pd.DataFrame(rq1_all_results)
        rq1_df.to_csv(rq1_results_path, index=False)
        print(f"Results saved after {dataset_name} seed {seed}")

# ---- Final summary ----
rq1_df = pd.DataFrame(rq1_all_results)
rq1_df.to_csv(rq1_results_path, index=False)

print("\n===== RQ1 SUMMARY ACROSS ALL SEEDS =====")
print(rq1_df.groupby("dataset")[["accuracy", "f1_macro", "f1_sarcastic"]]
      .agg(["mean", "std"]).round(4))

RQ1: IN-DOMAIN BERT — NO CONTEXT
Resuming — 30 runs already completed
Completed: [('Headlines', 42), ('Reddit', 42), ('SemEval', 42), ('Headlines', 123), ('Reddit', 123), ('SemEval', 123), ('Headlines', 456), ('Reddit', 456), ('SemEval', 456), ('Headlines', 789), ('Reddit', 789), ('SemEval', 789), ('Headlines', 1011), ('Reddit', 1011), ('SemEval', 1011), ('Headlines', 1213), ('Reddit', 1213), ('SemEval', 1213), ('Headlines', 1415), ('Reddit', 1415), ('SemEval', 1415), ('Headlines', 1617), ('Reddit', 1617), ('SemEval', 1617), ('Headlines', 1819), ('Reddit', 1819), ('SemEval', 1819), ('Headlines', 2021), ('Reddit', 2021), ('SemEval', 2021)]

SEED 42
Skipping Headlines seed 42 — already done
Skipping Reddit seed 42 — already done
Skipping SemEval seed 42 — already done

SEED 123
Skipping Headlines seed 123 — already done
Skipping Reddit seed 123 — already done
Skipping SemEval seed 123 — already done

SEED 456
Skipping Headlines seed 456 — already done
Skipping Reddit seed 456 — already d

## 5. RQ2 — Effect of Context (Reddit)

Two BERT models are trained on Reddit:

- One using only the target comment (`text`)
- One using the parent comment prepended to the target (`text_with_context`)

The no-context result is reused directly from RQ1 — no retraining needed.

This answers: **Does incorporating contextual information improve sarcasm detection?**

In [9]:
print("=" * 60)
print("RQ2: EFFECT OF CONTEXT — REDDIT ONLY")
print("=" * 60)

# ---- Load existing results if resuming ----
rq2_results_path = os.path.join(MODELS_DIR, "rq2_bert_results.csv")

if os.path.exists(rq2_results_path):
    rq2_df_existing = pd.read_csv(rq2_results_path)
    rq2_all_results = rq2_df_existing.to_dict("records")
    completed_rq2   = [
        (r["model"], r["seed"]) for r in rq2_all_results
    ]
    print(f"Resuming — {len(completed_rq2)} runs already completed")
    print(f"Completed: {completed_rq2}")
else:
    rq2_all_results = []
    completed_rq2   = []
    print("Starting fresh run")

# ---- Load RQ1 results if not in memory ----
rq1_results_path = os.path.join(MODELS_DIR, "rq1_bert_results.csv")
if "rq1_all_results" not in dir() or len(rq1_all_results) == 0:
    rq1_df       = pd.read_csv(rq1_results_path)
    rq1_all_results = rq1_df.to_dict("records")
    print(f"Loaded RQ1 results: {len(rq1_all_results)} rows")

for seed in SEEDS:
    print(f"\n{'='*40}")
    print(f"SEED {seed}")
    print(f"{'='*40}")

    set_seed(seed)

    rd_train = rd_train_full.sample(20000, random_state=seed)
    rd_test  = rd_test_full.sample(5000,  random_state=seed)

    # Condition A — No context

    if ("BERT (no context)", seed) not in completed_rq2:
        rq1_reddit_seed = [
            r for r in rq1_all_results
            if r["dataset"] == "Reddit" and r["seed"] == seed
        ][0].copy()
        rq1_reddit_seed["model"] = "BERT (no context)"
        rq2_all_results.append(rq1_reddit_seed)
        completed_rq2.append(("BERT (no context)", seed))

        # Save immediately
        rq2_df = pd.DataFrame(rq2_all_results)
        rq2_df.to_csv(rq2_results_path, index=False)
        print(f"\n--- Condition A: No context (reused from RQ1) ---")
        print(f"  F1 macro: {rq1_reddit_seed['f1_macro']:.4f} — saved")
    else:
        print(f"\n--- Condition A: No context seed {seed} — skipping")

    # Condition B — Dual Encoder
    if ("BERT (dual encoder)", seed) not in completed_rq2:
        print("\n--- Condition B: Dual Encoder ---")

        set_seed(seed)

        tokenizer_dual = BertTokenizer.from_pretrained("bert-base-uncased")

        train_dataset_dual = DualEncoderDataset(
            contexts=rd_train["context"].fillna("").astype(str),
            replies=rd_train["text"].astype(str),
            labels=rd_train["label"].values,
            tokenizer=tokenizer_dual
        )
        test_dataset_dual = DualEncoderDataset(
            contexts=rd_test["context"].fillna("").astype(str),
            replies=rd_test["text"].astype(str),
            labels=rd_test["label"].values,
            tokenizer=tokenizer_dual
        )

        train_loader_dual = DataLoader(
            train_dataset_dual, batch_size=16, shuffle=True
        )
        test_loader_dual = DataLoader(
            test_dataset_dual, batch_size=16, shuffle=False
        )

        dual_model = DualEncoderBert()
        dual_model = train_dual_encoder(dual_model, train_loader_dual, epochs=3)

        y_pred_dual = predict_dual_encoder(dual_model, test_loader_dual)
        y_true_dual = rd_test["label"].values

        result_dual = evaluate(y_true_dual, y_pred_dual,
                               dataset_name="Reddit",
                               model_name="BERT (dual encoder)")
        result_dual["seed"] = seed
        rq2_all_results.append(result_dual)
        completed_rq2.append(("BERT (dual encoder)", seed))

        # Save dual encoder checkpoint
        dual_save_path = os.path.join(
            MODELS_DIR, f"bert_reddit_dual_encoder_seed{seed}"
        )
        os.makedirs(dual_save_path, exist_ok=True)
        torch.save(
            dual_model.state_dict(),
            os.path.join(dual_save_path, "model.pt")
        )
        tokenizer_dual.save_pretrained(dual_save_path)
        rd_train.to_csv(
            os.path.join(dual_save_path, "rd_train_seed.csv"), index=False
        )
        rd_test.to_csv(
            os.path.join(dual_save_path, "rd_test_seed.csv"), index=False
        )
        print(f"Dual encoder checkpoint saved: {dual_save_path}")

        # Save results immediately after condition B
        rq2_df = pd.DataFrame(rq2_all_results)
        rq2_df.to_csv(rq2_results_path, index=False)
        print(f"Results saved after seed {seed}")

    else:
        print(f"\n--- Condition B: Dual encoder seed {seed} — skipping")

# ---- Final summary ----
rq2_df = pd.DataFrame(rq2_all_results)
rq2_df.to_csv(rq2_results_path, index=False)

print("\n===== RQ2 SUMMARY ACROSS ALL SEEDS =====")
print(rq2_df.groupby("model")[["accuracy", "f1_macro", "f1_sarcastic"]]
      .agg(["mean", "std"]).round(4))

RQ2: EFFECT OF CONTEXT — REDDIT ONLY
Resuming — 20 runs already completed
Completed: [('BERT (no context)', 42), ('BERT (dual encoder)', 42), ('BERT (no context)', 123), ('BERT (dual encoder)', 123), ('BERT (no context)', 456), ('BERT (dual encoder)', 456), ('BERT (no context)', 789), ('BERT (dual encoder)', 789), ('BERT (no context)', 1011), ('BERT (dual encoder)', 1011), ('BERT (no context)', 1213), ('BERT (dual encoder)', 1213), ('BERT (no context)', 1415), ('BERT (dual encoder)', 1415), ('BERT (no context)', 1617), ('BERT (dual encoder)', 1617), ('BERT (no context)', 1819), ('BERT (dual encoder)', 1819), ('BERT (no context)', 2021), ('BERT (dual encoder)', 2021)]

SEED 42

--- Condition A: No context seed 42 — skipping

--- Condition B: Dual encoder seed 42 — skipping

SEED 123

--- Condition A: No context seed 123 — skipping

--- Condition B: Dual encoder seed 123 — skipping

SEED 456

--- Condition A: No context seed 456 — skipping

--- Condition B: Dual encoder seed 456 — skippi

## 6. RQ3 — Cross-Domain Generalisation

Each fine-tuned model from RQ1 is evaluated directly on the two
datasets it was never trained on — no additional training.

The drop in F1 score compared to in-domain performance (RQ1)
measures how much domain shift hurts generalisation.

This answers: **How well do sarcasm detection models generalise across domains?**

Cross-domain pairs evaluated:
- Headlines → Reddit, SemEval
- Reddit → Headlines, SemEval
- SemEval → Headlines, Reddit

In [5]:
print("=" * 60)
print("RQ3: CROSS-DOMAIN GENERALISATION")
print("=" * 60)

# ---- Load RQ1 results if not already in memory ----
rq1_results_path = os.path.join(MODELS_DIR, "rq1_bert_results.csv")
if "rq1_all_results" not in dir() or len(rq1_all_results) == 0:
    rq1_df = pd.read_csv(rq1_results_path)
    rq1_all_results = rq1_df.to_dict("records")
    print(f"Loaded RQ1 results: {len(rq1_all_results)} rows")
else:
    print(f"RQ1 results already in memory: {len(rq1_all_results)} rows")

# ---- Load existing RQ3 results if resuming ----
rq3_results_path = os.path.join(MODELS_DIR, "rq3_bert_results.csv")

if os.path.exists(rq3_results_path):
    rq3_df_existing = pd.read_csv(rq3_results_path)
    rq3_all_results = rq3_df_existing.to_dict("records")
    completed = [
        (r["train_domain"], r["test_domain"], r["seed"])
        for r in rq3_all_results
    ]
    print(f"Resuming — {len(completed)} pairs already completed")
else:
    rq3_all_results = []
    completed = []
    print("Starting fresh run")

cross_domain_pairs = [
    ("Headlines", "Reddit"),
    ("Headlines", "SemEval"),
    ("Reddit",    "Headlines"),
    ("Reddit",    "SemEval"),
    ("SemEval",   "Headlines"),
    ("SemEval",   "Reddit"),
]

for seed in SEEDS:
    print(f"\n{'='*40}")
    print(f"SEED {seed}")
    print(f"{'='*40}")

    set_seed(seed)

    # Subsample Reddit test with this seed — same as RQ1
    rd_test = rd_test_full.sample(5000, random_state=seed)

    # Fixed test sets for this seed
    test_sets = {
        "Headlines": (hl_test,  "text"),
        "Reddit":    (rd_test,  "text"),
        "SemEval":   (se_test,  "text"),
    }

    for train_domain, test_domain in cross_domain_pairs:

        # Skip if already completed
        if (train_domain, test_domain, seed) in completed:
            print(f"Skipping {train_domain} → {test_domain} seed {seed}")
            continue

        print(f"\n--- Train: {train_domain} → Test: {test_domain} ---")

        # Load fine-tuned model saved during RQ1 for this seed
        model_path = os.path.join(
            MODELS_DIR, f"bert_{train_domain.lower()}_seed{seed}"
        )

        if not os.path.exists(model_path):
            print(f"WARNING: Model not found at {model_path} — skipping")
            continue

        model     = BertForSequenceClassification.from_pretrained(model_path)
        tokenizer = BertTokenizer.from_pretrained(model_path)

        test_df, text_col = test_sets[test_domain]
        test_enc          = tokenize_data(test_df[text_col], tokenizer)
        test_loader       = make_dataloader(
            test_enc, test_df["label"], shuffle=False
        )

        y_pred = predict_bert(model, test_loader)
        y_true = test_df["label"].values

        result = evaluate(y_true, y_pred,
                          dataset_name=f"{train_domain} → {test_domain}",
                          model_name="BERT (cross-domain)")
        result["seed"]         = seed
        result["train_domain"] = train_domain
        result["test_domain"]  = test_domain
        rq3_all_results.append(result)

        # Save after every pair in case of crash
        rq3_df = pd.DataFrame(rq3_all_results)
        rq3_df.to_csv(rq3_results_path, index=False)
        print(f"  F1 macro: {result['f1_macro']:.4f} — saved")

# ---- Final summary ----
rq3_df = pd.DataFrame(rq3_all_results)
rq3_df.to_csv(rq3_results_path, index=False)

print("\n===== RQ3 SUMMARY ACROSS ALL SEEDS =====")
print(rq3_df.groupby("dataset")[["accuracy", "f1_macro", "f1_sarcastic"]]
      .agg(["mean", "std"]).round(4))

print("\n===== RQ3 BY TRAIN DOMAIN =====")
print(rq3_df.groupby("train_domain")[["accuracy", "f1_macro", "f1_sarcastic"]]
      .agg(["mean", "std"]).round(4))

RQ3: CROSS-DOMAIN GENERALISATION
Loaded RQ1 results: 30 rows
Resuming — 60 pairs already completed

SEED 42
Skipping Headlines → Reddit seed 42
Skipping Headlines → SemEval seed 42
Skipping Reddit → Headlines seed 42
Skipping Reddit → SemEval seed 42
Skipping SemEval → Headlines seed 42
Skipping SemEval → Reddit seed 42

SEED 123
Skipping Headlines → Reddit seed 123
Skipping Headlines → SemEval seed 123
Skipping Reddit → Headlines seed 123
Skipping Reddit → SemEval seed 123
Skipping SemEval → Headlines seed 123
Skipping SemEval → Reddit seed 123

SEED 456
Skipping Headlines → Reddit seed 456
Skipping Headlines → SemEval seed 456
Skipping Reddit → Headlines seed 456
Skipping Reddit → SemEval seed 456
Skipping SemEval → Headlines seed 456
Skipping SemEval → Reddit seed 456

SEED 789
Skipping Headlines → Reddit seed 789
Skipping Headlines → SemEval seed 789
Skipping Reddit → Headlines seed 789
Skipping Reddit → SemEval seed 789
Skipping SemEval → Headlines seed 789
Skipping SemEval → Redd

## 7. Full Results Summary

All results consolidated across RQ1, RQ2, and RQ3.

In [10]:
# ============================================================
# 7. FULL SUMMARY
# ============================================================
print("=" * 60)
print("FULL RESULTS SUMMARY — MEAN ± STD ACROSS 10 SEEDS")
print("=" * 60)

print("\n--- RQ1: In-Domain Performance ---")
print(rq1_df.groupby("dataset")[["accuracy", "f1_macro", "f1_sarcastic"]]
      .agg(["mean", "std"]).round(4))

print("\n--- RQ2: Effect of Context (Reddit) ---")
print(rq2_df.groupby("model")[["accuracy", "f1_macro", "f1_sarcastic"]]
      .agg(["mean", "std"]).round(4))

print("\n--- RQ3: Cross-Domain Generalisation ---")
print(rq3_df.groupby("dataset")[["accuracy", "f1_macro", "f1_sarcastic"]]
      .agg(["mean", "std"]).round(4))

# Save combined for evaluation notebook
all_results = {
    "rq1": rq1_df,
    "rq2": rq2_df,
    "rq3": rq3_df
}
for name, df in all_results.items():
    df.to_csv(os.path.join(MODELS_DIR, f"{name}_bert_results.csv"), index=False)

print("\nAll results saved to models/ directory.")
print("Load these CSVs in 07_evaluation.ipynb for statistical analysis.")

FULL RESULTS SUMMARY — MEAN ± STD ACROSS 10 SEEDS

--- RQ1: In-Domain Performance ---
          accuracy         f1_macro         f1_sarcastic        
              mean     std     mean     std         mean     std
dataset                                                         
Headlines   0.9303  0.0014   0.9300  0.0014       0.9252  0.0016
Reddit      0.7226  0.0034   0.7225  0.0034       0.7250  0.0042
SemEval     0.9529  0.0040   0.9529  0.0039       0.9534  0.0040

--- RQ2: Effect of Context (Reddit) ---
                    accuracy         f1_macro         f1_sarcastic        
                        mean     std     mean     std         mean     std
model                                                                     
BERT (dual encoder)   0.7213  0.0059   0.7211  0.0059       0.7264  0.0071
BERT (no context)     0.7226  0.0034   0.7225  0.0034       0.7250  0.0042

--- RQ3: Cross-Domain Generalisation ---
                    accuracy         f1_macro         f1_sarcastic